## 1.0 Realization rotation and append prep

In [12]:
import os
import re
import numpy as np

# ================== CONFIG ==================
input_dir  = r"C:\Users\xo\CNN_3D_python\20250805_newcnn model\realization"
# Two common name patterns you’ve used before; script will try both:
patterns   = [
    ("realization_", ".dat"),
]
n_blocks   = 4045           # 0..3604 inclusive
shape      = (20, 20, 20)    # (Z, X, Y)
dtype      = np.float32
save_dir   = r"C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version"
# ============================================

os.makedirs(save_dir, exist_ok=True)

def find_existing_pattern():
    """Return (prefix, ext) that actually exists in the folder, otherwise None."""
    for prefix, ext in patterns:
        test_path = os.path.join(input_dir, f"{prefix}{0}{ext}")
        if os.path.isfile(test_path):
            return prefix, ext
    # fallback: try to sniff any file like *_<number>.dat and infer prefix/ext
    for name in os.listdir(input_dir):
        m = re.match(r"^(.*?)(\d+)(\.[A-Za-z0-9]+)$", name)
        if m and os.path.isfile(os.path.join(input_dir, name)):
            return m.group(1), m.group(3)
    return None

found = find_existing_pattern()
if not found:
    raise FileNotFoundError(
        "Could not find any realization files in the folder. "
        "Expected names like 'realization_0.dat' "
    )
prefix, ext = found
print(f"Using detected file pattern: {prefix}# {ext}")

# ------------ Load originals (Z-direction) ------------
blocks = []
missing = []
for i in range(n_blocks):
    fname = os.path.join(input_dir, f"{prefix}{i}{ext}")
    if not os.path.isfile(fname):
        missing.append(i)
        # You can choose to raise or continue. We'll raise to avoid silent gaps.
        continue
    arr = np.loadtxt(fname, dtype=dtype)
    if arr.size != np.prod(shape):
        raise ValueError(f"{fname} has {arr.size} values, expected {np.prod(shape)} for {shape}.")
    arr = arr.reshape(shape)  # (Z, X, Y)
    blocks.append(arr)

if missing:
    raise FileNotFoundError(f"Missing files for indices: {missing[:10]}{' ...' if len(missing)>10 else ''}")

data_z = np.stack(blocks, axis=0)  # (N, Z, X, Y)
print("Z-dir stack:", data_z.shape)

# Rotate so original **Y** becomes new Z  → this is your X-dir stack now
data_x = np.rot90(data_z, k=1, axes=(1, 3))  # (Z,Y) plane
print("X-dir stack:", data_x.shape)

# Rotate so original **X** becomes new Z  → this is your Y-dir stack now
data_y = np.rot90(data_z, k=1, axes=(1, 2))  # (Z,X) plane
print("Y-dir stack:", data_y.shape)

# ------------ Save outputs ------------
# Concatenate (Z + X + Y)
augmented = np.concatenate([data_z, data_x, data_y], axis=0)
all_rotation_data = np.expand_dims(np.asarray(augmented), 1)
print("Augmented stack:", all_rotation_data.shape)
np.save(os.path.join(save_dir, "realizations_3dirs_12135.npy"), all_rotation_data)

print("Saved:")
print("  - realizations_3dirs_12135.npy")



Using detected file pattern: realization_# .dat
Z-dir stack: (4045, 20, 20, 20)
X-dir stack: (4045, 20, 20, 20)
Y-dir stack: (4045, 20, 20, 20)
Augmented stack: (12135, 1, 20, 20, 20)
Saved:
  - realizations_3dirs_12135.npy


## 2.0 Prep dataset training output

In [13]:
import os
import re
import shutil
from pathlib import Path

# ================== SETTINGS ==================
NEW_ROOT        = r"C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version"
SRC_MODEL_ROOT  = r"C:\Users\xo\CNN_3D_python\20250805_newcnn model"

# Keep the same folder names at the destination
FEATURE_FOLDERS = [
    "soil strength input",
    "pore pressure input",
    "plastic strain input",
    "volumetric strain input",
]

# Only process CSVs (set to True to include other extensions too)
CSV_ONLY  = True
INCLUDE_OTHER_EXTS = []  # e.g., [".tab", ".dat"] if needed

# Indexing (append X to Z, Y to X)
N_PER_DIR = 4045  
OFFSETS   = {"z": 0, "x": N_PER_DIR, "y": 2 * N_PER_DIR}

DRY_RUN   = False     # set False to actually copy
OVERWRITE = False    # keep False for safety
# ==============================================

# File name patterns
PAT_DIR  = re.compile(r"^(ss)_(z|x|y)_(qa|vol|pp|ps)_table_realization_(\d+)(\.[A-Za-z0-9]+)?$", re.IGNORECASE)
PAT_NEWZ = re.compile(r"^new(z)_(qa|vol|pp|ps)_table_realization_(\d+)(\.[A-Za-z0-9]+)?$", re.IGNORECASE)

def ensure_dir(p: Path):
    if not p.exists() and not DRY_RUN:
        p.mkdir(parents=True, exist_ok=True)

def safe_copy(src: Path, dst: Path):
    if dst.exists() and not OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite existing file: {dst}")
    if not DRY_RUN:
        ensure_dir(dst.parent)
        shutil.copy2(src, dst)

def want_ext(name: str) -> bool:
    if CSV_ONLY:
        return name.lower().endswith(".csv")
    if INCLUDE_OTHER_EXTS:
        return any(name.lower().endswith(ext.lower()) for ext in INCLUDE_OTHER_EXTS + [".csv"])
    return True  # fallback

def process_features():
    total = 0
    for folder in FEATURE_FOLDERS:
        src_dir = Path(SRC_MODEL_ROOT) / folder
        dst_dir = Path(NEW_ROOT) / folder
        ensure_dir(dst_dir)

        if not src_dir.exists():
            print(f"WARNING: missing folder → {src_dir}")
            continue

        ops = []
        per_dir_counts = {"z":0,"x":0,"y":0}
        for root, _, files in os.walk(src_dir):
            root_p = Path(root)
            for name in files:
                if not want_ext(name):
                    continue
                src_path = root_p / name

                # Normalize "newz_" → treat as Z
                m_newz = PAT_NEWZ.match(name)
                if m_newz:
                    dir_letter, metric, idx, ext = m_newz.groups()
                    dir_letter = "z"
                    metric = metric.lower()
                    i = int(idx); ext = ext or ".csv"
                    if not (0 <= i < N_PER_DIR): continue
                    new_idx = i + OFFSETS[dir_letter]
                    new_name = f"ss_{metric}_table_realization_{new_idx}{ext}"
                    ops.append((src_path, dst_dir / new_name))
                    per_dir_counts[dir_letter]+=1
                    continue

                # Normal pattern ss_[z|x|y]_{metric}_table_realization_{i}.ext
                m = PAT_DIR.match(name)
                if not m:
                    continue
                prefix, dir_letter, metric, idx, ext = m.groups()
                dir_letter = dir_letter.lower()
                metric = metric.lower()
                i = int(idx); ext = ext or ".csv"
                if dir_letter not in OFFSETS: continue
                if not (0 <= i < N_PER_DIR): continue

                new_idx = i + OFFSETS[dir_letter]
                # Drop the direction in the new name
                new_name = f"{prefix.lower()}_{metric}_table_realization_{new_idx}{ext}"
                ops.append((src_path, dst_dir / new_name))
                per_dir_counts[dir_letter]+=1

        print(f"[{folder}] plan: {len(ops)} CSV files "
              f"(z:{per_dir_counts['z']} x:{per_dir_counts['x']} y:{per_dir_counts['y']})")
        for s, d in ops[:20]:
            print(f"  {s} -> {d}")
        if len(ops) > 20:
            print(f"  ... and {len(ops)-20} more")

        for s, d in ops:
            safe_copy(s, d)
        total += len(ops)
    return total

def main():
    new_root = Path(NEW_ROOT)
    print(f"{'[DRY RUN] ' if DRY_RUN else ''}Creating: {new_root}")
    ensure_dir(new_root)
    total_feats = process_features()
    print(f"\nDone. CSV files copied/renamed: {total_feats}"
          f"{' (DRY RUN — nothing written)' if DRY_RUN else ''}")

if __name__ == "__main__":
    main()


Creating: C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version
[soil strength input] plan: 12135 CSV files (z:4045 x:4045 y:4045)
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_0.csv -> C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\soil strength input\ss_qa_table_realization_4045.csv
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_1.csv -> C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\soil strength input\ss_qa_table_realization_4046.csv
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_10.csv -> C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\soil strength input\ss_qa_table_realization_4055.csv
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_100.csv -> C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\soil strength input\ss_qa_table_realization_4145.csv
  C:\Users\xo

[plastic strain input] plan: 12135 CSV files (z:4045 x:4045 y:4045)
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_0.csv -> C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\plastic strain input\ss_ps_table_realization_4045.csv
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_1.csv -> C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\plastic strain input\ss_ps_table_realization_4046.csv
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_10.csv -> C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\plastic strain input\ss_ps_table_realization_4055.csv
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_100.csv -> C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\plastic strain input\ss_ps_table_realization_4145.csv
  C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain inpu


Done. CSV files copied/renamed: 48540


In [14]:
## replaceing the first point of the pore pressure from (0,0) to (0,981)
import os
import csv
import re
from pathlib import Path

# === SETTINGS ===
PP_DIR = Path(r"C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\pore pressure input")
MAX_IDX = 12135
TARGET_PP = 981.0  # value to replace first point with
FILE_PATTERN = re.compile(r"^ss_pp_table_realization_(\d+)\.csv$", re.IGNORECASE)

def replace_first_point(pp_dir: Path, target_value: float = TARGET_PP, idx_max: int = MAX_IDX):
    count = 0
    for file in pp_dir.iterdir():
        if not file.is_file():
            continue
        m = FILE_PATTERN.match(file.name)
        if not m:
            continue
        idx = int(m.group(1))
        if not (0 <= idx <= idx_max):
            continue

        try:
            with open(file, "r", newline="") as f:
                sample = f.read(4096)
                f.seek(0)
                try:
                    dialect = csv.Sniffer().sniff(sample)
                    delim = dialect.delimiter
                except Exception:
                    delim = ","  # fallback if no delimiter found
                reader = list(csv.reader(f, delimiter=delim))

            # Locate the first numeric data line (skip headers)
            first_data_row = None
            for i, row in enumerate(reader):
                if len(row) < 2:
                    continue
                try:
                    float(row[0]); float(row[1])
                    first_data_row = i
                    break
                except ValueError:
                    continue

            if first_data_row is None:
                print(f"[SKIP] No numeric row found in {file.name}")
                continue

            # Replace first pair (0, 0) → (0, 981)
            row = reader[first_data_row]
            row[0] = "0"
            row[1] = str(target_value)

            with open(file, "w", newline="") as f:
                writer = csv.writer(f, delimiter=delim)
                writer.writerows(reader)

            count += 1

        except Exception as e:
            print(f"[ERROR] {file.name}: {e}")

    print(f"✅ Updated first row to (0, {target_value}) in {count} file(s).")

if __name__ == "__main__":
    replace_first_point(PP_DIR)



✅ Updated first row to (0, 981.0) in 12135 file(s).


## 3.0 Compact into single output npy file

In [15]:
import os
import pandas as pd
import numpy as np

# ===== CONFIG =====
BASE_DIR = r"C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version"

features = ['plastic strain', 'pore pressure', 'soil strength', 'volumetric strain']
metric_code = {
    'plastic strain':    'ps',
    'pore pressure':     'pp',
    'soil strength':     'qa',
    'volumetric strain': 'vol',
}

# counts & indexing
N_PER_DIR = 4045                    # indices 0..3944 (adjust if needed)
OFFSETS   = {
    'z': 0,
    'x': N_PER_DIR,
    'y': 2 * N_PER_DIR
}
ORDER     = ['z', 'x', 'y']          # concatenate Z, then X, then Y

# CSV extraction
ROWS  = 65
COL   = 1
DTYPE = np.float32

# mode: single-channel only (for (N,4,65) final)
THREE_CHANNELS = False               # keep False for your current model
OUT_FILENAME   = "combined_output_12135.npy"   # change name if needed
# ===================


def unified_path(folder, metric, idx):
    return os.path.join(folder, f"ss_{metric}_table_realization_{idx}.csv")


def load_vec(fp):
    df = pd.read_csv(fp, header=0)
    if COL >= df.shape[1]:
        raise IndexError(f"{fp}: column {COL} not found; csv has {df.shape[1]} columns.")
    rows = min(ROWS, len(df))
    vec = df.iloc[:rows, COL].to_numpy(dtype=DTYPE)
    if rows < ROWS:
        pad_val = vec[-1] if rows > 0 else 0.0
        vec = np.pad(vec, (0, ROWS - rows), mode='constant', constant_values=pad_val)
    return vec


def build_feature_array(feature_name):
    """
    Returns:
        arr: shape (N_total, 1, 65) for this feature
    """
    metric = metric_code[feature_name]
    folder = os.path.join(BASE_DIR, f"{feature_name} input")

    if THREE_CHANNELS:
        raise NotImplementedError("This combined script is set up for single-channel mode only.")

    # (N_total, 1, 65): concatenate Z block, then X block, then Y block
    vecs = []
    for d in ORDER:
        for i in range(N_PER_DIR):
            idx = OFFSETS[d] + i
            fp  = unified_path(folder, metric, idx)
            if not os.path.exists(fp):
                raise FileNotFoundError(f"Missing CSV: {fp}")
            vecs.append(load_vec(fp))

    arr = np.expand_dims(np.asarray(vecs, dtype=DTYPE), 1)  # (N_total, 1, 65)
    print(f"{feature_name}: built array with shape {arr.shape}")
    return arr


def main():
    # Build per-feature arrays in memory
    arrays = []
    for feat in features:
        arr = build_feature_array(feat)
        # basic sanity checks
        if arr.ndim != 3 or arr.shape[1] != 1:
            raise ValueError(f"{feat}: expected shape (N,1,65); got {arr.shape}")
        arrays.append(arr)

    # Sanity check same N and L across features
    N_set = {a.shape[0] for a in arrays}
    L_set = {a.shape[2] for a in arrays}
    if len(N_set) != 1 or len(L_set) != 1:
        raise ValueError(f"Mismatch across features: N={N_set}, L={L_set}")

    # Concatenate along feature dimension → (N, 4, 65)
    combined = np.concatenate(arrays, axis=1)
    out_path = os.path.join(BASE_DIR, OUT_FILENAME)

    with open(out_path, "wb"):
        np.save(out_path, combined)

    print(f"\nSaved combined output → {out_path}  shape={combined.shape}")


if __name__ == "__main__":
    main()


plastic strain: built array with shape (12135, 1, 65)
pore pressure: built array with shape (12135, 1, 65)
soil strength: built array with shape (12135, 1, 65)
volumetric strain: built array with shape (12135, 1, 65)

Saved combined output → C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\combined_output_12135.npy  shape=(12135, 4, 65)


## 4.0 split the dataset into training and testing

In [16]:
import numpy as np
from pathlib import Path

def train_test_split_with_ranges(
    x_path: str,
    y_path: str = None,         # optional paired labels
    n_total: int = 12135,
    test_ranges=((3000,3124), (7045,7169), (11090,11214)),  # inclusive
):
    """
    Test set  = union of the absolute index ranges above.
    Train set = all remaining indices.

    Saves:
        *_train.npy
        *_test.npy
        idx_train_big.npy
        idx_test.npy
    """

    # ---- Load arrays ----
    X = np.load(x_path)
    assert X.shape[0] == n_total, f"Expected {n_total}, got {X.shape[0]}"

    Y = None
    if y_path is not None:
        Y = np.load(y_path)
        assert Y.shape[0] == n_total, "X/Y length mismatch."

    # ---- Build test index set from inclusive ranges ----
    test_idx = []
    for a, b in test_ranges:
        assert 0 <= a <= b < n_total, f"Bad range [{a},{b}]"
        test_idx.extend(range(a, b + 1))

    test_idx = sorted(set(test_idx))
    all_idx  = np.arange(n_total, dtype=int)

    # Train = complement of test
    test_mask = np.zeros(n_total, dtype=bool)
    test_mask[test_idx] = True

    train_idx = all_idx[~test_mask].tolist()
    test_idx  = all_idx[test_mask].tolist()

    # ---- Save helper ----
    def _save_subset(base_path: str, idx, suffix: str, arr):
        p = Path(base_path)
        out = p.with_name(p.stem + f"_{suffix}.npy")
        np.save(out, arr[np.array(idx, dtype=int)])
        return out

    # ---- Save X splits ----
    x_train_p = _save_subset(x_path, train_idx, "train", X)
    x_test_p  = _save_subset(x_path,  test_idx, "test",  X)

    # ---- Save Y splits ----
    if Y is not None:
        y_train_p = _save_subset(y_path, train_idx, "train", Y)
        y_test_p  = _save_subset(y_path,  test_idx, "test",  Y)

    # ---- Save index files ----
    output_dir = Path(x_path).parent
    np.save(output_dir / "idx_train_big.npy", np.array(train_idx, dtype=int))
    np.save(output_dir / "idx_test.npy",      np.array(test_idx,  dtype=int))

    # ---- Report ----
    print(f"Test ranges (abs, inclusive): {test_ranges}")
    print(f"Counts → train: {len(train_idx)}, test: {len(test_idx)}")
    print(f"Saved X → {x_train_p.name}, {x_test_p.name}")
    if Y is not None:
        print(f"Saved Y → {y_train_p.name}, {y_test_p.name}")
    print("Saved idx_train_big.npy, idx_test.npy")


# ==== Your exact case ====

train_test_split_with_ranges(
    x_path="realizations_3dirs_12135.npy",
    y_path=r"C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\combined_output_12135.npy",
    n_total=12135,
    test_ranges=((3000,3124), (7045,7169), (11090,11214)),
)


Test ranges (abs, inclusive): ((3000, 3124), (7045, 7169), (11090, 11214))
Counts → train: 11760, test: 375
Saved X → realizations_3dirs_12135_train.npy, realizations_3dirs_12135_test.npy
Saved Y → combined_output_12135_train.npy, combined_output_12135_test.npy
Saved idx_train_big.npy, idx_test.npy


## 5.0 K-fold for the training dataset

In [17]:
import numpy as np
from pathlib import Path

from sklearn.model_selection import StratifiedKFold


def make_kfold_split(
    x_train_path: str,
    y_train_path: str,
    idx_train_big_path: str = "idx_train_big.npy",
    n_splits: int = 5,
    fold_id: int = 0,
    seed: int = 123,
    num_bins: int = 6,
):
    """
    Create one K-fold (train/val) split from the BIG TRAIN pool only.

    Inputs:
        x_train_path      : e.g. "realizations_3dirs_10815_train.npy"
        y_train_path      : e.g. "combined_output_10815_train.npy"
        idx_train_big_path: "idx_train_big.npy" from the first split
        n_splits          : number of folds (e.g. 5)
        fold_id           : which fold to export (0 .. n_splits-1)
        seed              : RNG seed for reproducibility
        num_bins          : bins for qa_peak stratification

    Outputs:
        Saves:
            realizations_3dirs_10815_train_fold{fold_id}.npy
            realizations_3dirs_10815_val_fold{fold_id}.npy
            combined_output_10815_train_fold{fold_id}.npy
            combined_output_10815_val_fold{fold_id}.npy
            idx_train_fold{fold_id}.npy
            idx_val_fold{fold_id}.npy
    """

    # ---- Load big-train data (10440 samples) ----
    X = np.load(x_train_path)    # (N_big, ...)
    Y = np.load(y_train_path)    # (N_big, 4, 65)
    assert X.shape[0] == Y.shape[0], "X/Y length mismatch in big train set"
    n_big = X.shape[0]

    # absolute indices of big-train w.r.t. original 10815
    idx_big_train = np.load(idx_train_big_path)
    assert idx_big_train.shape[0] == n_big, "idx_train_big length mismatch"

    # ---- Compute qa_peak per sample (channel 2) ----
    qa = Y[:, 2, :]            # (N_big, 65)
    qa_peak = qa.max(axis=1)   # (N_big,)

    # ---- Build bins for stratification ----
    # num_bins=6 -> 7 quantiles
    quantiles = np.linspace(0.0, 1.0, num_bins + 1)
    edges = np.quantile(qa_peak, quantiles)
    edges[0]  -= 1e-6
    edges[-1] += 1e-6

    # Turn qa_peak into discrete labels for StratifiedKFold
    # bins 0..num_bins-1
    bin_labels = np.digitize(qa_peak, edges[1:-1])  # shape (N_big,)

    # ---- Stratified K-Fold ----
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # Just extract the requested fold_id
    for fold, (train_rel, val_rel) in enumerate(skf.split(X, bin_labels)):
        if fold != fold_id:
            continue

        # relative indices into X/Y
        train_rel = np.array(train_rel, dtype=int)
        val_rel   = np.array(val_rel, dtype=int)

        # absolute indices (w.r.t original 10815)
        idx_train_abs = idx_big_train[train_rel]
        idx_val_abs   = idx_big_train[val_rel]

        # subset arrays
        X_train_fold = X[train_rel]
        X_val_fold   = X[val_rel]
        Y_train_fold = Y[train_rel]
        Y_val_fold   = Y[val_rel]

        # ---- Save ----
        x_p = Path(x_train_path)
        y_p = Path(y_train_path)
        out_dir = x_p.parent

        # remove the trailing "_train" from stem if present
        x_base = x_p.stem.replace("_train", "")
        y_base = y_p.stem.replace("_train", "")

        x_train_out = out_dir / f"{x_base}_train_fold{fold_id}.npy"
        x_val_out   = out_dir / f"{x_base}_val_fold{fold_id}.npy"
        y_train_out = out_dir / f"{y_base}_train_fold{fold_id}.npy"
        y_val_out   = out_dir / f"{y_base}_val_fold{fold_id}.npy"

        np.save(x_train_out, X_train_fold)
        np.save(x_val_out,   X_val_fold)
        np.save(y_train_out, Y_train_fold)
        np.save(y_val_out,   Y_val_fold)

        np.save(out_dir / f"idx_train_fold{fold_id}.npy", idx_train_abs)
        np.save(out_dir / f"idx_val_fold{fold_id}.npy",   idx_val_abs)

        print(f"[Fold {fold_id}]")
        print(f"  Inner train size: {len(train_rel)}")
        print(f"  Inner val size  : {len(val_rel)}")
        print(f"  Saved X: {x_train_out.name}, {x_val_out.name}")
        print(f"  Saved Y: {y_train_out.name}, {y_val_out.name}")
        print(f"  Saved idx_train_fold{fold_id}.npy, idx_val_fold{fold_id}.npy")
        break


# ====== CALL IT FOR YOUR CASE (all 5 folds) ======
for fold_id in range(5):  # 0,1,2,3,4
    make_kfold_split(
        x_train_path=r"C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\realizations_3dirs_12135_train.npy",
        y_train_path=r"C:\Users\xo\CNN_3D_python\20251118_final 3dcnn version\combined_output_12135_train.npy",
        idx_train_big_path="idx_train_big.npy",
        n_splits=5,
        fold_id=fold_id,
        seed=123,
        num_bins=6,
    )


[Fold 0]
  Inner train size: 9408
  Inner val size  : 2352
  Saved X: realizations_3dirs_12135_train_fold0.npy, realizations_3dirs_12135_val_fold0.npy
  Saved Y: combined_output_12135_train_fold0.npy, combined_output_12135_val_fold0.npy
  Saved idx_train_fold0.npy, idx_val_fold0.npy
[Fold 1]
  Inner train size: 9408
  Inner val size  : 2352
  Saved X: realizations_3dirs_12135_train_fold1.npy, realizations_3dirs_12135_val_fold1.npy
  Saved Y: combined_output_12135_train_fold1.npy, combined_output_12135_val_fold1.npy
  Saved idx_train_fold1.npy, idx_val_fold1.npy
[Fold 2]
  Inner train size: 9408
  Inner val size  : 2352
  Saved X: realizations_3dirs_12135_train_fold2.npy, realizations_3dirs_12135_val_fold2.npy
  Saved Y: combined_output_12135_train_fold2.npy, combined_output_12135_val_fold2.npy
  Saved idx_train_fold2.npy, idx_val_fold2.npy
[Fold 3]
  Inner train size: 9408
  Inner val size  : 2352
  Saved X: realizations_3dirs_12135_train_fold3.npy, realizations_3dirs_12135_val_fold3.n